In [44]:
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

1.获取数据源

In [45]:
# \s+：空白字符
df = pd.read_table('userlostprob.txt', sep='\s+')
print(df.shape)
print(df.columns)
print(df.head())

(689945, 51)
Index(['label', 'sampleid', 'd', 'arrival', 'iforderpv_24h',
       'decisionhabit_user', 'historyvisit_7ordernum',
       'historyvisit_totalordernum', 'hotelcr', 'ordercanceledprecent',
       'landhalfhours', 'ordercanncelednum', 'commentnums', 'starprefer',
       'novoters', 'consuming_capacity', 'historyvisit_avghotelnum',
       'cancelrate', 'historyvisit_visit_detailpagenum', 'delta_price1',
       'price_sensitive', 'hoteluv', 'businessrate_pre', 'ordernum_oneyear',
       'cr_pre', 'avgprice', 'lowestprice', 'firstorder_bu',
       'customereval_pre2', 'delta_price2', 'commentnums_pre',
       'customer_value_profit', 'commentnums_pre2', 'cancelrate_pre',
       'novoters_pre2', 'novoters_pre', 'ctrip_profits', 'deltaprice_pre2_t1',
       'lowestprice_pre', 'uv_pre', 'uv_pre2', 'lowestprice_pre2',
       'lasthtlordergap', 'businessrate_pre2', 'cityuvs', 'cityorders',
       'lastpvgap', 'cr', 'sid', 'visitnum_oneyear', 'h'],
      dtype='object')
   label  sam

2.数据清洗

①缺失值

计算各字段缺失值比例

In [46]:
# 各字段缺失比例计算
na_rate = (len(df)-df.count())/len(df)
na_rate_order = na_rate.sort_values(ascending=False)
na_rate_order

historyvisit_7ordernum              0.879824
historyvisit_visit_detailpagenum    0.554698
firstorder_bu                       0.453590
decisionhabit_user                  0.441332
historyvisit_totalordernum          0.439774
historyvisit_avghotelnum            0.437816
delta_price1                        0.366405
delta_price2                        0.365529
customer_value_profit               0.363539
ctrip_profits                       0.354750
ordernum_oneyear                    0.350918
ordercanceledprecent                0.350918
ordercanncelednum                   0.350918
lasthtlordergap                     0.350918
avgprice                            0.337250
cr                                  0.336330
consuming_capacity                  0.327719
price_sensitive                     0.327719
starprefer                          0.326190
businessrate_pre                    0.298646
deltaprice_pre2_t1                  0.212720
lastpvgap                           0.140775
visitnum_o

缺失值处理（按列名处理）

In [47]:
# 缺失数据量超过80%，舍弃该特征
dropfeatures = ['historyvisit_7ordernum'] 

# 缺失值填充0
fillfeatureswith0 = [
                  'historyvisit_totalordernum', # 近1年用户历史订单数
                  'ordercanceledprecent', # 用户一年内取消订单率
                  'historyvisit_visit_detailpagenum'  # 7天内访问酒店详情页数
                  ]

# 缺失值20%-50%之间的用-999代替
fillNauWith999 = ['ordercanncelednum',  # 用户一年内取消订单数
                  'landhalfhours',  # 24小时内登陆时长
                  'historyvisit_avghotelnum',  # 近3个月用户历史日均访问酒店数
                  'ordernum_oneyear',  # 年订单数
                  'avgprice',  # 平均价格
                  'customer_value_profit',  # 客户近一年的价值
                  'ctrip_profits',  # 客户价值
                  'lasthtlordergap',  # 一年内距离上次下单时长
                  'lastpvgap',  # 一年内距上次访问时长
                  'cr'  # 用户转化率
                  ]

# 缺失值较少的用平均值代替
fillNauWithMean = ['commentnums',  # 酒店评论数
                   'novoters',  # 酒店当前评论人数
                   'cancelrate',  # 当前酒店历史取消率
                   'hoteluv',  # 当前酒店历史UV
                   'hotelcr',  # 当前酒店历史转化率
                   'cr_pre',  # 24小时历史浏览次数最多酒店历史综合人气指标
                   'lowestprice',  # 当前酒店可定最低价
                   'lowestprice_pre2',  # 24小时访问酒店可预定最低价
                   'customereval_pre2',  # 24小时历史浏览酒店客户评分均值
                   'commentnums_pre',  # 24小时历史浏览次数最多酒店点评数
                   'commentnums_pre2',  # 24小时历史浏览酒店点评数均值
                   'novoters_pre2',  # 24小时历史浏览酒店评分人数均值
                   'novoters_pre',  # 24小时历史浏览次数最多酒店评分人数
                   'deltaprice_pre2_t1',  # 24小时内已访问酒店价格与对手价差均值
                   'lowestprice_pre',  # 24小时内已访问次数最多酒店可订最低价
                   'uv_pre',  # 24小时历史浏览次数最多酒店历史uv
                   'uv_pre2',  # 24小时历史浏览酒店历史uv均值
                   'businessrate_pre',  # 24小时历史浏览次数最多酒店商务属性指数
                   'businessrate_pre2',  # 24小时内已访问酒店商务属性指数均值
                   'cityuvs',  # 昨日访问当前城市同入住日期的app uv数
                   'cityorders',  # 昨日提交当前城市同入住日期的app订单数
                   'visitnum_oneyear',  # 年访问次数
                   'delta_price1',  # 用户偏好价格-24小时浏览最多酒店价格
                   'delta_price2',   # 用户偏好价格-24小时浏览酒店平均价格
                   'firstorder_bu', # 首单bu
                   'decisionhabit_user' # 用户决策习惯
                   ]

# 缺失值用中位数代替
fillNauWithMedian = ['starprefer',  # 星级偏好
                  'consuming_capacity',  # 消费能力指数
                  'price_sensitive',  # 价格敏感指数
                  'cancelrate_pre'   # 24小时内已访问次数最多酒店历史取消率
                  ]

In [48]:
def missValueProcess(data):
    # 删除缺失值较多的列
    data = data.drop(columns=dropfeatures)
    # 缺失值填充0
    for col in fillfeatureswith0:
        data[col] = data[col].fillna(0)
    # 缺失值填充-999
    for col in fillNauWith999:
        data[col] = data[col].fillna(-999)
    # 缺失值填充均值
    for col in fillNauWithMean:
        fillvalue1 = (data[col].mean())
        data[col] = data[col].fillna(fillvalue1)
    # 缺失值填充中位数
    for col in fillNauWithMedian:
        fillvalue2 = (data[col].median())
        data[col] = data[col].fillna(fillvalue2)
        
    return data

In [49]:
df = missValueProcess(df)
print(df.shape)
print(df.head())

(689945, 50)
   label  sampleid           d     arrival  iforderpv_24h  decisionhabit_user  \
0      0     24636  2016-05-18  2016-05-18              0            5.317048   
1      1     24637  2016-05-18  2016-05-18              0            5.317048   
2      0     24641  2016-05-18  2016-05-19              0            5.317048   
3      0     24642  2016-05-18  2016-05-18              0            5.317048   
4      1     24644  2016-05-18  2016-05-19              0            5.317048   

   historyvisit_totalordernum  hotelcr  ordercanceledprecent  landhalfhours  \
0                         0.0     1.04                   0.0           22.0   
1                         0.0     1.06                   0.0            0.0   
2                         0.0     1.05                   0.0            3.0   
3                         0.0     1.01                   0.0            2.0   
4                         0.0     1.00                   0.0            0.0   

   ...  lowestprice_pre2 

②异常值处理

In [50]:
df.head()

,label,sampleid,d,arrival,iforderpv_24h,decisionhabit_user,historyvisit_totalordernum,hotelcr,ordercanceledprecent,landhalfhours,...,lowestprice_pre2,lasthtlordergap,businessrate_pre2,cityuvs,cityorders,lastpvgap,cr,sid,visitnum_oneyear,h
0,0,24636,2016-05-18,2016-05-18,0,5.317048,0.0,1.04,0.0,22.0,...,615.0,-999.0,0.290000,12.880,3.14700,-999.0,-999.0,7,18551.846682,12
1,1,24637,2016-05-18,2016-05-18,0,5.317048,0.0,1.06,0.0,0.0,...,513.0,-999.0,0.530000,17.933,4.91300,-999.0,-999.0,33,18551.846682,14
2,0,24641,2016-05-18,2016-05-19,0,5.317048,0.0,1.05,0.0,3.0,...,382.0,-999.0,0.600000,3.993,0.76000,-999.0,-999.0,10,18551.846682,19
3,0,24642,2016-05-18,2016-05-18,0,5.317048,0.0,1.01,0.0,2.0,...,203.0,-999.0,0.180000,3.220,0.66000,-999.0,-999.0,8,18551.846682,16
4,1,24644,2016-05-18,2016-05-19,0,5.317048,0.0,1.00,0.0,0.0,...,84.0,-999.0,0.368237,0.013,2.25325,-999.0,-999.0,1,18551.846682,21


In [7]:
# 查看最小值为负值的特征
df_min = df.min().iloc[4:]
print(df_min[df_min<0])

landhalfhours                -999.0
ordercanncelednum            -999.0
historyvisit_avghotelnum     -999.0
delta_price1               -99879.0
ordernum_oneyear             -999.0
avgprice                     -999.0
lowestprice                    -3.0
delta_price2               -43344.0
customer_value_profit        -999.0
ctrip_profits                -999.0
deltaprice_pre2_t1          -2296.0
lasthtlordergap              -999.0
lastpvgap                    -999.0
cr                           -999.0
dtype: object


In [8]:
ano_values1 = ['delta_price1','delta_price2','lowestprice']
ano_values2 = ['customer_value_profit','ctrip_profits']
# 异常值填充中位数
for col in ano_values1:
    df.loc[df[col]<0,col] = df[col].median()
# 异常值填充0
for col in ano_values2:
    df.loc[df[col]<0,col] = 0

③构造新特征

时间特征处理

In [9]:
df.dtypes

label                                 int64
sampleid                              int64
d                                    object
arrival                              object
iforderpv_24h                         int64
decisionhabit_user                  float64
historyvisit_totalordernum          float64
hotelcr                             float64
ordercanceledprecent                float64
landhalfhours                       float64
ordercanncelednum                   float64
commentnums                         float64
starprefer                          float64
novoters                            float64
consuming_capacity                  float64
historyvisit_avghotelnum            float64
cancelrate                          float64
historyvisit_visit_detailpagenum    float64
delta_price1                        float64
price_sensitive                     float64
hoteluv                             float64
businessrate_pre                    float64
ordernum_oneyear                

In [10]:
# 将访问日期d字段转换为日期格式（原本是string类型的）---预定时间
df['d'] = pd.to_datetime(df['d'], format = '%Y-%m-%d')

In [11]:
# 将入住日期arrivald字段转换为日期格式（原本是string类型的）
df['arrival'] = pd.to_datetime(df['arrival'], format='%Y-%m-%d')

构造新特征：星期几/是否工作日/预定时间与入住时间间隔（天）

In [12]:
def createNewFeatures(data):
    # 星期几（1-7的数字）
    data['week'] = data['arrival'].map(lambda x: datetime.isoweekday(x))
    # 是否是工作日（1:是 0:否）
    data['isweekday'] = data['week'].apply(lambda x: 1 if x<6 else 0)
    # 入住时间-预定时间(天粒度)
    data['gap'] = (pd.to_datetime(data['arrival'])-pd.to_datetime(data['d'])).map(lambda x : int(x / np.timedelta64(1, 'D')))
    # 删除特征
    data = data.drop(['arrival', 'd'],axis=1)
    
    return data

In [13]:
df = createNewFeatures(df)
print(df.shape)
print(df.head())

(689945, 51)
   label  sampleid  iforderpv_24h  decisionhabit_user  \
0      0     24636              0            5.317048   
1      1     24637              0            5.317048   
2      0     24641              0            5.317048   
3      0     24642              0            5.317048   
4      1     24644              0            5.317048   

   historyvisit_totalordernum  hotelcr  ordercanceledprecent  landhalfhours  \
0                         0.0     1.04                   0.0           22.0   
1                         0.0     1.06                   0.0            0.0   
2                         0.0     1.05                   0.0            3.0   
3                         0.0     1.01                   0.0            2.0   
4                         0.0     1.00                   0.0            0.0   

   ordercanncelednum  commentnums  ...  cityuvs  cityorders  lastpvgap     cr  \
0             -999.0  1089.000000  ...   12.880     3.14700     -999.0 -999.0   
1      

连续特征离散化

In [14]:
# decisionhabit_user（对用户决策习惯）
def deal_decisionhabit_user(x):
    if x<10:
        return 0
    elif x<30:
        return 1
    else:
        return 2

In [15]:
# starprefer（星级偏好）
def deal_starprefer(x):
    if x<40:
        return 0
    elif x<80:
        return 1
    else:
        return 2

In [16]:
# avgprice（平均价格）
def deal_avgprice(x):
    if x<400:
        return 0
    elif x<1000:
        return 1
    else:
        return 2

In [17]:
# consuming_capacity（消费能力指数进行离散化处理）
def deal_consuming_capacity(x):
    if x<40:
        return 0
    elif x<80:
        return 1
    else:
        return 2

In [18]:
# 调用函数进行连续特征离散化
df['decisionhabit_user'] = df['decisionhabit_user'].map(deal_decisionhabit_user)
df['starprefer'] = df['starprefer'].map(deal_starprefer)
df['avgprice'] = df['avgprice'].map(deal_avgprice)
df['consuming_capacity'] = df['consuming_capacity'].map(deal_consuming_capacity)
print(df.shape)
print(df.head())

(689945, 51)
   label  sampleid  iforderpv_24h  decisionhabit_user  \
0      0     24636              0                   0   
1      1     24637              0                   0   
2      0     24641              0                   0   
3      0     24642              0                   0   
4      1     24644              0                   0   

   historyvisit_totalordernum  hotelcr  ordercanceledprecent  landhalfhours  \
0                         0.0     1.04                   0.0           22.0   
1                         0.0     1.06                   0.0            0.0   
2                         0.0     1.05                   0.0            3.0   
3                         0.0     1.01                   0.0            2.0   
4                         0.0     1.00                   0.0            0.0   

   ordercanncelednum  commentnums  ...  cityuvs  cityorders  lastpvgap     cr  \
0             -999.0  1089.000000  ...   12.880     3.14700     -999.0 -999.0   
1      

④筛选特征

In [19]:
# 相关性
label_corr = df.corr()['label'].abs().sort_values(ascending=False)
print(label_corr)

label                               1.000000
gap                                 0.153750
historyvisit_totalordernum          0.146961
ordercanncelednum                   0.139953
businessrate_pre2                   0.131205
ordernum_oneyear                    0.122950
hotelcr                             0.115490
businessrate_pre                    0.114748
cr_pre                              0.114067
iforderpv_24h                       0.110308
cityuvs                             0.099925
cityorders                          0.096938
customer_value_profit               0.085603
ctrip_profits                       0.080742
h                                   0.077728
landhalfhours                       0.073261
uv_pre2                             0.068420
uv_pre                              0.061440
lowestprice_pre2                    0.061344
ordercanceledprecent                0.054683
hoteluv                             0.052039
lowestprice_pre                     0.047706
lowestpric

In [36]:
feature = list(label_corr[label_corr>0.04].index)
print(len(feature))
print(feature)

23
['label', 'gap', 'historyvisit_totalordernum', 'ordercanncelednum', 'businessrate_pre2', 'ordernum_oneyear', 'hotelcr', 'businessrate_pre', 'cr_pre', 'iforderpv_24h', 'cityuvs', 'cityorders', 'customer_value_profit', 'ctrip_profits', 'h', 'landhalfhours', 'uv_pre2', 'uv_pre', 'lowestprice_pre2', 'ordercanceledprecent', 'hoteluv', 'lowestprice_pre', 'lowestprice']


In [37]:
df_m = df[feature]
print(df_m.shape)
print(df_m.head())

(689945, 23)
   label  gap  historyvisit_totalordernum  ordercanncelednum  \
0      0    0                         0.0             -999.0   
1      1    0                         0.0             -999.0   
2      0    1                         0.0             -999.0   
3      0    0                         0.0             -999.0   
4      1    1                         0.0             -999.0   

   businessrate_pre2  ordernum_oneyear  hotelcr  businessrate_pre  cr_pre  \
0           0.290000            -999.0     1.04          0.250000    1.03   
1           0.530000            -999.0     1.06          0.510000    1.07   
2           0.600000            -999.0     1.05          0.610000    1.12   
3           0.180000            -999.0     1.01          0.372717    1.01   
4           0.368237            -999.0     1.00          0.372717    1.03   

   iforderpv_24h  ...  ctrip_profits   h  landhalfhours  uv_pre2   uv_pre  \
0              0  ...            0.0  12           22.0   74.9

模型一：决策树

划分数据集

In [38]:
X_train, X_test, y_train, y_test = train_test_split(df_m.iloc[:, 1:], df_m['label'], 
                                                    test_size=0.2, random_state=188)

构建训练模型

In [39]:
clf = tree.DecisionTreeClassifier()
clf = clf.fit(X_train, y_train)

准确度

In [40]:
clf.score(X_test, y_test)

0.832261991897905

模型二：随机森林

In [41]:
# 建立随机森林
rfc = RandomForestClassifier(n_estimators=100, random_state=90)

In [42]:
# 用交叉验证计算得分
score_pre = cross_val_score(rfc, df.iloc[:, 1:], df['label'], cv=3).mean()

In [43]:
print(score_pre)

0.9062722323582659
